## 1. Config

In [1]:
# =========================
# CONFIG
# =========================

CONFIG = {
    "size": 100000,

    "word_max_features": 15000,
    "char_max_features": 15000,

    "min_df": 2,

    "word_ngram_range": (1, 2),
    "char_ngram_range": (2, 6),

    "lr": 0.3,
    "epochs": 40,
}

print("CONFIGURATION:")
for k, v in CONFIG.items():
    print(f"{k}: {v}")

CONFIGURATION:
size: 100000
word_max_features: 15000
char_max_features: 15000
min_df: 2
word_ngram_range: (1, 2)
char_ngram_range: (2, 6)
lr: 0.3
epochs: 40


## 2. Imports

In [2]:
import numpy as np
import pandas as pd

from src.dataloader.dataloader import SentenceDataModule
from src.tfidf.tfidf import TfIdfVectorizerNumpy
from src.models.logreg import SoftmaxLogReg
from src.train.train_logreg import save_model

## 3. Load dataset

In [3]:
from pathlib import Path

BASE_DIR = Path().resolve().parent  # move out of notebooks/
DATA_PATH = BASE_DIR / "datasets" / "records_long.json"

print("\nLoading dataset...")

dm = SentenceDataModule(
    record_path=str(DATA_PATH),
    size=CONFIG["size"],
    split=(70, 20, 10),
)

train_samples = dm.get_train_loader().samples
val_samples = dm.get_val_loader().samples
test_samples = dm.get_test_loader().samples

print("Train size:", len(train_samples))
print("Val size:", len(val_samples))
print("Test size:", len(test_samples))


Loading dataset...
Sample in dataset 100000
Train size: 70000
Val size: 20000
Test size: 10000


## 4. Extract text + labels

In [4]:
X_train_text = [s["text"] for s in train_samples]
y_train = np.array([s["model"] for s in train_samples])

label_to_id = {
    "human": 0,
    "chatgpt": 1,
    "mistral": 2,
    "gemma": 3,
    "llama": 4
}

id_to_label = {v: k for k, v in label_to_id.items()}

total = len(y_train)

unique, counts = np.unique(y_train, return_counts=True)

print("\nTraining distribution:")
for u, c in zip(unique, counts):
    pct = (c / total) * 100
    print(f"{id_to_label[u]:10s}: {c:6d} ({pct:5.2f}%)")

X_val_text = [s["text"] for s in val_samples]
y_val = np.array([s["model"] for s in val_samples])

X_test_text = [s["text"] for s in test_samples]
y_test = np.array([s["model"] for s in test_samples])

print("\nExample sample:")
print(X_train_text[0])
print("Label:", y_train[0])


Training distribution:
human     :  14087 (20.12%)
chatgpt   :  13990 (19.99%)
mistral   :  13982 (19.97%)
gemma     :  13934 (19.91%)
llama     :  14007 (20.01%)

Example sample:
HiWelcome to healthcaremagicI have gone through your query and understand your concern.This may be muscular pain. Its treatment is rest to joint and analgesic such as ibuprofen for pain relief. Vitamin B and C is helpful in recovery. As your whole arm aches you are also advised to rule out vitamin D deficiency. If found low you are advised to take vitamin D3. You can discuss with your doctor about it. Hope your query get answered. If you have any clarification then don't hesitate to write to us. I will be happy to help you.Wishing you a good health.Take care.
Label: 0


## 5.1 WORD TF-IDF

In [5]:
print("\nBuilding WORD TF-IDF...")

word_vectorizer = TfIdfVectorizerNumpy(
    max_features=CONFIG["word_max_features"],
    min_df=CONFIG["min_df"],
    ngram_range=CONFIG["word_ngram_range"],
    analyzer="word",
)

X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)
X_test_word = word_vectorizer.transform(X_test_text)

print("Word TF-IDF shape:", X_train_word.shape)


Building WORD TF-IDF...
Word TF-IDF shape: (70000, 15000)


## 5.2 Char TF-IDF

In [6]:
print("\nBuilding CHAR TF-IDF...")

char_vectorizer = TfIdfVectorizerNumpy(
    max_features=CONFIG["char_max_features"],
    min_df=CONFIG["min_df"],
    ngram_range=CONFIG["char_ngram_range"],
    analyzer="char",
)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)
X_test_char = char_vectorizer.transform(X_test_text)

print("Char TF-IDF shape:", X_train_char.shape)


Building CHAR TF-IDF...
Char TF-IDF shape: (70000, 15000)


## 7. Combine features and add length

In [7]:
print("\nCombining features...")

X_train = np.hstack([X_train_word, X_train_char])
X_val = np.hstack([X_val_word, X_val_char])
X_test = np.hstack([X_test_word, X_test_char])

print("Combined shape:", X_train.shape)

print("\nAdding length feature...")

length_train = np.array([len(t) for t in X_train_text]).reshape(-1, 1) / 200
length_val = np.array([len(t) for t in X_val_text]).reshape(-1, 1) / 200
length_test = np.array([len(t) for t in X_test_text]).reshape(-1, 1) / 200

X_train = np.hstack([X_train, length_train])
X_val = np.hstack([X_val, length_val])
X_test = np.hstack([X_test, length_test])

print("Final feature shape:", X_train.shape)


Combining features...
Combined shape: (70000, 30000)

Adding length feature...
Final feature shape: (70000, 30001)


## 8. Train model and save it

In [8]:
print("\nTraining model...")

clf = SoftmaxLogReg(
    input_dim=X_train.shape[1],
    num_classes=5,
    lr=CONFIG["lr"],
)

clf.fit(
    X_train,
    y_train,
    X_val=X_val,
    y_val=y_val,
    epochs=CONFIG["epochs"],
)

MODEL_PATH = Path("../models/subm1-g5-MEI-A.pkl")

save_model(
    model=clf,
    word_vectorizer=word_vectorizer,
    char_vectorizer=char_vectorizer,
    path=MODEL_PATH,
    config=CONFIG,
    label_to_id=label_to_id,
    save_full_model=True
)


Training model...
Epoch 001 | train_loss=1.4242 train_acc=0.5304 | val_loss=1.4286 val_acc=0.5234
Epoch 002 | train_loss=1.3159 train_acc=0.5751 | val_loss=1.3228 val_acc=0.5681
Epoch 003 | train_loss=1.2446 train_acc=0.5295 | val_loss=1.2523 val_acc=0.5246
Epoch 004 | train_loss=1.1806 train_acc=0.6375 | val_loss=1.1893 val_acc=0.6305
Epoch 005 | train_loss=1.1428 train_acc=0.6427 | val_loss=1.1530 val_acc=0.6352
Epoch 006 | train_loss=1.1242 train_acc=0.6025 | val_loss=1.1346 val_acc=0.5964
Epoch 007 | train_loss=1.0672 train_acc=0.7104 | val_loss=1.0782 val_acc=0.7017
Epoch 008 | train_loss=1.0442 train_acc=0.6963 | val_loss=1.0559 val_acc=0.6878
Epoch 009 | train_loss=1.0209 train_acc=0.7237 | val_loss=1.0332 val_acc=0.7152
Epoch 010 | train_loss=0.9982 train_acc=0.7404 | val_loss=1.0111 val_acc=0.7331
Epoch 011 | train_loss=0.9815 train_acc=0.7501 | val_loss=0.9947 val_acc=0.7389
Epoch 012 | train_loss=0.9667 train_acc=0.7527 | val_loss=0.9807 val_acc=0.7421
Epoch 013 | train_los

## 9. Evaluate

In [9]:
print("\nEvaluating on test set...")

preds = clf.predict(X_test)
acc = np.mean(preds == y_test)

print("Test accuracy:", acc)


Evaluating on test set...
Test accuracy: 0.8138


## 10. Test the given subm1.csv

In [10]:
print("\nLoading CSV...")

df = pd.read_csv(
    str(BASE_DIR) + "/tests/TestData/subm1.csv",
    sep=";",
)

texts = df["Text"].tolist()

print("Number of samples:", len(texts))




print("\nBuilding features for CSV...")

X_word = word_vectorizer.transform(texts)
X_char = char_vectorizer.transform(texts)

X = np.hstack([X_word, X_char])

length = np.array([len(t) for t in texts]).reshape(-1, 1) / 200
X = np.hstack([X, length])

print("CSV feature shape:", X.shape)

label_to_id = {
    "human": 0,
    "chatgpt": 1,
    "mistral": 2,
    "gemma": 3,
    "llama": 4
}

id_to_company = {
    0: "Human",
    1: "OpenAI",
    2: "Anthropic",
    3: "Google",
    4: "Meta"
}




print("\nRunning predictions...")

preds = clf.predict(X)

df["Label"] = [id_to_company[p] for p in preds]

print(df.head())




df.to_csv("subm1-g5-MEI-A.csv", index=False)
print("subm1-g5-MEI-A.csv")


Loading CSV...
Number of samples: 150

Building features for CSV...
CSV feature shape: (150, 30001)

Running predictions...
     ID                                               Text   Label
0  D2-1  A covalent bond is a chemical bond that involv...  OpenAI
1  D2-2  A covalent bond forms when two atoms share one...  OpenAI
2  D2-3  A covalent bond is a type of chemical bond whe...  OpenAI
3  D2-4  A covalent bond is a chemical bond that involv...  OpenAI
4  D2-5  Driven by exciting developments in the field o...  OpenAI
subm1-g5-MEI-A.csv
